# Beginner tour 8: Bugs, and the tests that catch them

Research code has bugs. The dangerous ones don't crash: they quietly give you **wrong
numbers** that look fine. On 2026-10-10 the project found three such bugs in its fusion model
(method [3]). Fixing them raised its results by up to **6 percentage points**. This notebook:

1. The two kinds of bugs, and Python's tools for each (`assert`, exceptions, tracebacks)
2. **Bug 1**: "UCB" was secretly greedy. Seen in the project's own saved results
3. **Bug 2**: ranking inside chunks instead of over the whole pool. Reproduced on real data
4. **Bug 3**: ties broken by position in the file
5. A crash bug: reading past the end of the pool
6. **Tests**: running the project's real tests, and watching them catch a bug we put back in
7. Writing a test of your own

About 1–2 minutes in total.

In [ ]:
import sys, json, inspect, traceback, io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tour import *

import run_experiment as rx
import surrogates
from molpal.acquirer.metrics import greedy, ucb
print("ready")

---
## 1. Two kinds of bugs

**Loud bugs** crash. Python stops and prints a **traceback**: the chain of function calls that
led to the error, with the **actual problem on the last line**. Annoying, but you know
something's wrong.

**Silent bugs** run to the end and print plausible numbers. You only catch them by
**checking**: comparing against something you know must be true.

🐍 **Python corner: exceptions and `try`/`except`.** An error is an **exception**.
`try: ... except SomeError:` catches it so the program can carry on. Here we cause one on
purpose and print its traceback without stopping the notebook:

In [ ]:
oracle = {"CCO": -2.6}
try:
    oracle["XYZ"]                        # a molecule that isn't there
except KeyError:
    traceback.print_exc(file=sys.stdout)
print("...and the notebook carries on")

Read tracebacks **from the bottom**: `KeyError: 'XYZ'` says exactly what went wrong.

🐍 **Python corner: `assert`.** `assert condition, "message"` does nothing when the condition is
true, and stops with an `AssertionError` when it's false. It's the simplest way to make a
silent bug loud: write down what **must** be true. For example, active learning should never
dock the same molecule twice:

In [ ]:
labeled = {3, 7, 11}
candidates = [1, 2, 4, 5]
assert not (labeled & set(candidates)), "a molecule is both docked and a candidate!"
print("check passed")

try:
    assert 2 + 2 == 5, "maths is broken"
except AssertionError as err:
    print("AssertionError:", err)

---
## 2. Bug 1: "UCB" that was secretly greedy

Remember from notebook 4: UCB ranks by `mu + 2 × sigma`. If a model returns `sigma = 0` for
every molecule, UCB **is** greedy, whatever you call it. Before the fix, the fusion model's
`predict()` returned `sigma = 0`, even though its description promised "rank disagreement as
uncertainty".

The evidence is still in the project's saved results. Here are the round-by-round recalls of
the old greedy run and the old "UCB" run on the full 2.1M pool:

In [ ]:
runs = AL_REPO / "runs"
def curve(name):
    return [round(r["topk_recall"], 3) for r in json.loads((runs / name / "history.json").read_text())]

print("old fusion, greedy:", curve("fusion_greedy_frac0.001"))
print("old fusion, 'ucb': ", curve("fusion_ucb_frac0.001"))

**Identical, round for round.** Two different strategies giving exactly the same numbers is a
red flag. It means the switch between them did nothing. Nobody noticed at first because each
number looked reasonable on its own.

The project still contains a model that returns `sigma = 0` on purpose (`BigFusionSurrogate`,
a simple rank vote with no uncertainty), so we can see the effect directly on real data:

In [ ]:
scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)           # flipped: higher = better
ALL = list(BACKBONES)
dims = {k: embeddings[k].shape[1] for k in ALL}
X = np.concatenate([embeddings[k] for k in ALL], axis=1)
train = np.random.default_rng(0).permutation(len(y))[:420]

def fitted(cls, dims=dims, X=X):
    torch.manual_seed(0)
    s = cls(dims=dims)
    with contextlib.redirect_stdout(io.StringIO()):
        s.fit(X[train], y[train])
    return s

# BigFusionSurrogate is hard-wired to the ORIGINAL three backbone names (its _KEYS list is
# ["grover", "molformer", "unimol"]) and splits the wide table in that order. So we hand it our
# three backbones under those names, glued in that order:
bf_dims = {"grover": dims["mhgged"], "molformer": dims["molformer"], "unimol": dims["unimol2"]}
X_bf = np.concatenate([embeddings["mhgged"], embeddings["molformer"], embeddings["unimol2"]], axis=1)

for cls, d, Xc in [(surrogates.BigFusionSurrogate, bf_dims, X_bf), (surrogates.EnsembleFusionSurrogate, dims, X)]:
    mu, sigma = fitted(cls, d, Xc).predict(Xc)
    same = set(np.argsort(-greedy(mu))[:210]) == set(np.argsort(-ucb(mu, sigma ** 2))[:210])
    print(f"{cls.__name__:24s} sigma all zero? {not sigma.any()!s:5s}   UCB batch == greedy batch? {same}")

With `sigma = 0`, the UCB batch is exactly the greedy batch. The fixed `EnsembleFusionSurrogate`
returns real uncertainty, and its UCB batch really differs.

(The hard-wired names are themselves a small lesson. Code written for one experiment often
bakes in assumptions, here the names of three backbones, that break when someone reuses it
later. The newer surrogates take whatever names `dims` gives them.)

---
## 3. Bug 2: ranking inside chunks

Notebook 7 showed that predictions are made in **chunks** of 50,000 molecules. The fusion model
works with **ranks** ("this molecule is the 3rd best"). Before the fix, ranks were computed
**inside each chunk**, so every chunk had its own "best molecule", its own "rank 1".

Why that's terrible: the real ENHITS file is **sorted by score**, best first. So chunk 1 holds
all the great molecules and the last chunk holds the worst. With per-chunk ranks, the worst
chunk's best molecule looks as good as chunk 1's best, and the batch gets spread evenly
across all chunks.

Let's reproduce it on real data. Sort our pool best-first (like ENHITS), cut it into 5 chunks,
and compare **one global ranking** with **per-chunk rankings**. We use the fixed code's
`predict_components` (each backbone's guesses) and `combine` (the ranking), calling `combine`
either once or once per chunk:

In [ ]:
fusion = fitted(surrogates.EnsembleFusionSurrogate)
components = fusion.predict_components(X)            # each backbone's guess for every molecule
best_first = np.argsort(-y)                          # sort the pool best-first, like the ENHITS file
comps_sorted = components[best_first]
chunk = len(y) // 5 + 1

global_rank = fusion.combine(comps_sorted)[0]                                   # the fix: rank once
per_chunk = np.concatenate([fusion.combine(comps_sorted[i:i + chunk])[0]        # the bug: rank per chunk
                            for i in range(0, len(y), chunk)])

for name, mu in [("ranked once (fixed)", global_rank), ("ranked per chunk (bug)", per_chunk)]:
    batch = np.argsort(-mu)[:210]                    # positions in the sorted pool: < 100 means a true best-100
    print(f"{name:24s} batch taken from chunks 1-5: {np.bincount(batch // chunk, minlength=5)}   "
          f"best-100 in batch: {(batch < 100).sum()}")

With one global ranking, the batch comes almost entirely from chunk 1, where the good
molecules are. With per-chunk ranking it's spread across **all five** chunks, including the
worst, and contains far fewer of the true best. On real runs this is a big chunk of why the
fix gained up to 6 points.

The fix in `run_experiment._chunked_get_means_and_vars`: predict each chunk's raw
per-backbone guesses (`predict_components`, safe per chunk), glue them together, and only
then call `combine` **once**. That's the `hasattr(surrogate, "combine")` branch you saw in
notebooks 5 and 7.

---
## 4. Bug 3: ties decided by position

On a GPU, predictions are computed with short 16-bit numbers for speed, so many molecules
get **exactly the same** guess. How many different values are there really?

In [ ]:
print(f"{len(np.unique(components[:, 0])):,} different guesses for {len(components):,} molecules")

Lots of ties. When values tie, *some* rule must order them. The old code used
`np.argsort(mu)[::-1]`, which gives the best rank in each tie group to the molecule that
appears **last in the file**. On a file sorted best-first, the last molecules are the worst.
So ties were systematically broken in favour of bad molecules.

The fix gives tied molecules the **same, averaged** rank (`_desc_ranks` in `surrogates.py`):

In [ ]:
tied = np.array([5.0, 7.0, 7.0, 7.0, 3.0])          # positions 1, 2, 3 tie

old = np.empty(len(tied)); old[np.argsort(tied)[::-1]] = np.arange(1, len(tied) + 1)
print("old ranks (1 = best):", old, "  <- among the tied ones, the LAST position wins rank 1")
print("new ranks (1 = best):", surrogates._desc_ranks(tied), "  <- tied molecules share rank 2")

---
## 5. A loud bug: reading past the end

Not all bugs are silent. One run (`fusionfix_*_frac0.004_seed42`) **crashed** with an
`IndexError`. The cause was in `EmbeddingMVEModel._get_X`, the speed trick from notebook 7:
"read one contiguous stretch starting at the first requested row". If the first requested
molecule was near the **end** of the pool, the stretch ran off the end of the table and came
back short.

Here's the old logic in a few lines, on a 10-row table:

In [ ]:
table = np.arange(20, dtype=np.float32).reshape(10, 2)       # 10 molecules, 2 numbers each
wanted = [8, 9, 0]                                          # first wanted row is near the end

def old_get_rows(table, wanted):
    base = wanted[0]
    block = np.array(table[base:base + len(wanted)])        # assumes rows base, base+1, base+2 exist
    wrong = [i for i, r in enumerate(wanted) if r != base + i]
    block[wrong] = table[[wanted[i] for i in wrong]]        # patch the ones that didn't line up
    return block

try:
    old_get_rows(table, wanted)
except IndexError:
    traceback.print_exc(limit=1, file=sys.stdout)

`table[8:11]` only has 2 rows (8 and 9), so patching "row 2 of the block" fails. The fix
checks for that case and falls back to plain indexing. The real, fixed method gives the right rows:

In [ ]:
from molpal.models.mvemodels import EmbeddingMVEModel
model = EmbeddingMVEModel(None, {"demo": table}, [f"mol_{i}" for i in range(10)])
print(model._get_X(["mol_8", "mol_9", "mol_0"]))

---
## 6. Tests: checks that run automatically

A **test** is a small function that checks one thing that must be true and fails loudly if it
isn't. After fixing the three fusion bugs, the project added `tests/test_rank_fusion.py`, so
the bugs can never come back unnoticed. Let's read one:

In [ ]:
sys.path.insert(0, str(AL_REPO / "tests"))
import test_rank_fusion as T
print(inspect.getsource(T.test_ties_are_not_broken_by_pool_position))

"If all 10 molecules are tied, they must all get the same score and zero uncertainty." Under
the old tie rule this would fail.

Normally you run tests with a tool called **pytest**: `python -m pytest tests/ -q` from the
`al-molecular` folder. It finds every function whose name starts with `test_`, runs it, and
reports pass/fail. (pytest isn't installed in the `py310` environment yet. `pip install
pytest` adds it.) We can do the same job by hand.

A few tests take an argument called `monkeypatch`, a pytest helper that **temporarily replaces**
something (for example the chunk size, so a test can force chunking on a small pool) and
puts it back afterwards. A tiny home-made version is enough:

🐍 **Python corner: `getattr` / `setattr`.** `getattr(obj, "name")` reads an attribute by its
name as text, and `setattr(obj, "name", value)` writes it. That's all monkeypatching is.

In [ ]:
class MonkeyPatch:
    # a minimal stand-in for pytest's monkeypatch fixture
    def __init__(self):
        self.saved = []
    def setattr(self, obj, name, value):
        self.saved.append((obj, name, getattr(obj, name)))
        setattr(obj, name, value)
    def undo(self):
        for obj, name, old in reversed(self.saved):
            setattr(obj, name, old)

def run_tests(module):
    results = {}
    for name, fn in vars(module).items():
        if not name.startswith("test_"):
            continue
        mp = MonkeyPatch()
        try:
            with contextlib.redirect_stdout(io.StringIO()):            # hide training printouts
                if "monkeypatch" in inspect.signature(fn).parameters:
                    fn(mp)
                else:
                    fn()
            results[name] = "PASS"
        except AssertionError as err:
            results[name] = "FAIL"
        finally:
            mp.undo()                                                  # always put things back
    return pd.Series(results, name="result")

run_tests(T)

All pass on the fixed code. Now the important part: **does the test suite actually catch the
bugs?** Put the old tie rule back (temporarily, with our monkeypatch) and run the tests again:

In [ ]:
def old_desc_ranks(mu):
    r = np.empty(len(mu))
    r[np.argsort(mu)[::-1]] = np.arange(1, len(mu) + 1)
    return r

mp = MonkeyPatch()
mp.setattr(surrogates, "_desc_ranks", old_desc_ranks)     # re-introduce bug 3
try:
    print(run_tests(T))
finally:
    mp.undo()                                             # fix it again
print("\nafter undo, the tie test passes again:", run_tests(T)["test_ties_are_not_broken_by_pool_position"])

The tie test turns red as soon as the bug comes back. That's the whole value of tests: you
change code months later, run the tests, and **any old bug you reintroduced announces itself**.

🐍 **Python corner: `try` / `finally`.** Code under `finally:` runs whether or not an error
happened. Here it guarantees the real `_desc_ranks` is restored even if a test crashes.

---
## 7. Write a test of your own

The `_get_X` crash from section 5 has no test in the repo. Let's write one. A good test is
short, checks one specific thing, and has a name that says what it checks:

In [ ]:
def test_get_x_handles_request_near_end_of_pool():
    table = np.arange(20, dtype=np.float32).reshape(10, 2)
    model = EmbeddingMVEModel(None, {"demo": table}, [f"mol_{i}" for i in range(10)])
    got = model._get_X(["mol_8", "mol_9", "mol_0"])
    assert got.shape == (3, 2), f"expected 3 rows, got {got.shape}"
    assert np.array_equal(got, table[[8, 9, 0]]), "rows came back wrong"

test_get_x_handles_request_near_end_of_pool()
print("PASS")

To add it to the project for real, you'd put that function in a file such as
`al-molecular/tests/test_get_x.py`, and `pytest` would pick it up automatically.

---
## Summary

| Bug | Kind | How it showed | The fix | Test |
|---|---|---|---|---|
| fusion UCB returned `sigma = 0` | silent | "greedy" and "UCB" runs identical, round for round | return rank disagreement as `sigma` | `test_ensemble_ucb_differs_from_greedy_and_matches_docstring` |
| ranks computed per chunk | silent | batches spread across good and bad chunks alike | collect all chunks, rank once (`combine`) | `test_ensemble_chunked_equals_whole`, `test_per_chunk_ranking_would_have_been_wrong` |
| ties favoured the end of the file | silent | systematic bias toward worse molecules | averaged ranks (`_desc_ranks`) | `test_ties_are_not_broken_by_pool_position` |
| `_get_X` read past the end | loud | `IndexError` crash | fall back to plain indexing | (yours, above) |

**Habits worth copying from this story:**
* When two things that *should* differ give identical numbers, suspect a bug.
* Write down what must be true (`assert`) in the code itself.
* After fixing a bug, add a test that would have caught it, and check the test **fails** on the old code.
* Re-run affected experiments. The project marked old results as affected and reran them as `runs/fusionfix_*`.

🐍 **Python you met:** exceptions, `try`/`except`/`finally`, tracebacks, `assert`, importing a
file by path, `inspect.signature`, `getattr`/`setattr`, monkeypatching, writing test functions.

## Try it yourself
1. Re-introduce bug 1: monkeypatch `surrogates.EnsembleFusionSurrogate.combine` with a version that returns `sigma` as zeros (`lambda self, comps: (..., np.zeros(len(comps)))`). Which test fails?
2. In section 3, try 10 chunks instead of 5. Does the per-chunk bug get worse?
3. Write a test that `random_recall` from notebook 2 is about 6% for 1,260 random molecules.

---
**That's the tour.** You've run every major part of al-molecular on real data: the data, the
experiment loop, the models, the acquisition rules, the classes that wire them together,
the adaptive follow-up project, the scale-up machinery, and the tests. The project's own write-ups
are now readable: start with `al-molecular/README.md` and `al-molecular/docs/adaptive_acq_results.md`.